# HapSleep — PoC multi-wearable: robustez del modelo generalizado · **exploratorio**

**Pregunta:** si el modelo se entrena con Fitbit (LifeSnaps) y se usa con otro reloj, ¿qué tanto se degrada, y qué diseño de
variables lo protege mejor? Como todavía no hay datos reales de otras marcas comparables, este notebook:

1. **Reúne de la literatura** cuánto se desvía cada marca frente a la polisomnografía (tiempo dormido, eficiencia, minutos
   despierto). Cada cifra viene de la página del estudio, verificada.
2. **Prueba aproximar los minutos sedentarios** con variables que Health Connect sí entrega (pasos, distancia, calorías, pulso).
3. **Crea datos sintéticos por marca:** toma a las 37 personas reales de LifeSnaps y **les aplica el sesgo típico de cada
   marca** a sus variables de sueño. Luego mide cuánto se degrada un modelo entrenado con Fitbit y comparan varios diseños de
   variables.

**Qué es y qué no es:** son copias distorsionadas de datos reales de Fitbit, con desviaciones tomadas de estudios contra
polisomnografía. Sirve para **comparar diseños de variables bajo un sesgo plausible**, no para afirmar cómo se comportará cada
marca. Los datos reales de los relojes que llegan (y una misma persona con dos relojes) son lo que lo confirmaría.

Las etiquetas siguen siendo el puntaje sustituto de Fitbit (37 personas).

In [1]:
import sys, json, warnings
from pathlib import Path
try:
    sys.stdout.reconfigure(encoding="utf-8")
except AttributeError:
    pass
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, GroupKFold
from sklearn.metrics import r2_score, roc_auc_score

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

LIFESNAPS = Path("../POC1/data/daily_fitbit_sema_df_unprocessed.csv")
COL_ID, LABEL_COL, SEED = "id", "sleep_points_percentage", 42
PSQI_CUT_LEVE, PSQI_CUT_ALTO, SCALE = 5, 10, 21
NUM = ["rmssd", "nremhr", "resting_hr", "spo2", "full_sleep_breathing_rate", "minutesToFallAsleep", "minutesAsleep", "minutesAwake",
       "sleep_efficiency", "steps", "sedentary_minutes", "distance", "calories", "bpm", "lightly_active_minutes",
       "moderately_active_minutes", "very_active_minutes", LABEL_COL]
daily = pd.read_csv(LIFESNAPS)
for c in NUM:
    daily[c] = pd.to_numeric(daily[c], errors="coerce")

has = daily.groupby(COL_ID)[LABEL_COL].apply(lambda s: s.notna().any())
ids37 = list(has[has].index)
y = pd.Series({p: 1 - g[LABEL_COL].mean() for p, g in daily[daily[COL_ID].isin(ids37)].groupby(COL_ID)}).reindex(ids37)
y_alto = (y * SCALE > PSQI_CUT_ALTO).astype(int)


def risk_level(v):
    s_ = v * SCALE
    return "Leve" if s_ <= PSQI_CUT_LEVE else ("Moderado" if s_ <= PSQI_CUT_ALTO else "Alto")


print(f"Personas con puntaje real: {len(ids37)} | con nivel real Alto: {int(y_alto.sum())}")

Personas con puntaje real: 37 | con nivel real Alto: 11


## 1. Cuánto se desvía cada marca de la polisomnografía (literatura)

Sesgo = **dispositivo menos polisomnografía**. Positivo en tiempo dormido y eficiencia significa que el reloj **sobreestima**;
negativo en WASO (minutos despierto tras dormirse) significa que **subestima el tiempo despierto**. Es el patrón que se ve en
todos: detectan casi todo el sueño (sensibilidad de 89 % a 95 %) pero detectan mal la vigilia (especificidad de 35 % a 52 %).

| Reloj | Estudio | Personas | Fuente |
|---|---|---|---|
| Fitbit Charge 5, Fitbit Sense, Withings ScanWatch, Garmin Vivosmart 4, Whoop 4.0, Apple Watch 8 | Schyvens et al. 2025, *SLEEP Advances* | 20 a 41 por reloj | [PMC12038347](https://pmc.ncbi.nlm.nih.gov/articles/PMC12038347/) |
| Xiaomi Mi Band 5 | Estudio de validación, *JMIR* 2023 | 45 | [PMC10238963](https://pmc.ncbi.nlm.nih.gov/articles/PMC10238963/) |
| Samsung Galaxy Watch 3 | Validación, *Journal of Sleep Medicine* 2023 | 32 | [jsm.230004](https://www.e-jsm.org/journal/view.php?doi=10.13078%2Fjsm.230004) |

**Lo que no se encontró:** una validación revisada por pares de **Amazfit** contra polisomnografía (solo páginas de terceros).
Como Amazfit y Xiaomi comparten ecosistema (Huami; en el dataset de Xiaomi los identificadores dicen "huami"), se les asigna el
**mismo sesgo como supuesto**, marcado como tal. Del Samsung solo se publicó el sesgo de tiempo dormido; eficiencia y WASO no
constan y se dejan sin sesgo adicional. Un meta-análisis de 24 estudios existe, pero no separa por marca y su signo no coincide
con el de los estudios individuales, así que no se usa.

In [2]:
# sesgo = dispositivo - polisomnografía. Fuente: páginas de cada estudio.
# Xiaomi: el estudio reporta el signo al revés (negativo = sobreestima); aquí se ajusta para coincidir con su resumen
# (sobreestima el tiempo dormido y subestima WASO en 31.44 min).
lit = pd.DataFrame([
    # reloj,                 TST,    SE,     WASO,   LoA_TST_low, LoA_TST_high
    ("Fitbit Charge 5",      11.12,  3.39,  -16.86,  -70.27,   92.50),
    ("Fitbit Sense",          6.31,  2.20,  -12.38, -124.70,  137.32),
    ("Withings ScanWatch",   39.87, 10.19,  -47.94,  -92.58,  172.31),
    ("Garmin Vivosmart 4",   38.44,  8.17,  -38.34,  -82.63,  159.51),
    ("Whoop 4.0",            24.46,  4.10,  -19.15, -101.09,  150.01),
    ("Apple Watch 8",        19.60,  4.44,  -21.23,  -38.35,   77.55),
    ("Xiaomi Mi Band 5",     29.54,  5.82,  -31.44, -171.72,  112.63),
    ("Samsung Galaxy Watch 3", 9.5,  np.nan, np.nan,  -81.2,   100.3),
], columns=["reloj", "sesgo TST (min)", "sesgo eficiencia (pts)", "sesgo WASO (min)", "LoA TST bajo", "LoA TST alto"]).set_index("reloj")
lit["DE de las diferencias TST (min)"] = ((lit["LoA TST alto"] - lit["LoA TST bajo"]) / 3.92).round(0)   # LoA = sesgo ± 1.96 DE
lit[["sesgo TST (min)", "sesgo eficiencia (pts)", "sesgo WASO (min)", "DE de las diferencias TST (min)"]]

,sesgo TST (min),sesgo eficiencia (pts),sesgo WASO (min),DE de las diferencias TST (min)
reloj,,,,
Fitbit Charge 5,11.12,3.39,-16.86,42.0
Fitbit Sense,6.31,2.20,-12.38,67.0
Withings ScanWatch,39.87,10.19,-47.94,68.0
Garmin Vivosmart 4,38.44,8.17,-38.34,62.0
Whoop 4.0,24.46,4.10,-19.15,64.0
Apple Watch 8,19.60,4.44,-21.23,30.0
Xiaomi Mi Band 5,29.54,5.82,-31.44,73.0
Samsung Galaxy Watch 3,9.50,NaN,NaN,46.0


### De sesgo absoluto a sesgo **relativo a Fitbit**
Nuestro modelo se entrena con datos de Fitbit, que ya trae su propio sesgo. Lo que importa al cambiar de marca es la
**diferencia respecto a Fitbit** (promedio de Charge 5 y Sense). El **ruido extra por noche** se estima con el ancho de los
límites de acuerdo: `raíz(DE_marca² − DE_Fitbit²)`. Es una aproximación (esa dispersión mezcla ruido entre noches y entre
personas), por eso se prueba con y sin ruido.

| Escenario | Basado en | Supuesto |
|---|---|---|
| Tipo Samsung | Galaxy Watch 3 | Solo consta el tiempo dormido |
| Tipo Xiaomi / Amazfit | Mi Band 5 | Amazfit igual que Xiaomi (mismo ecosistema) |
| Tipo Garmin | Vivosmart 4 | |
| Peor caso | Withings ScanWatch | El más alejado en la literatura revisada |

In [3]:
fit = lit.loc[["Fitbit Charge 5", "Fitbit Sense"]]
base = fit[["sesgo TST (min)", "sesgo eficiencia (pts)", "sesgo WASO (min)"]].mean()
sd_fit = fit["DE de las diferencias TST (min)"].mean()
SCEN_SRC = {"Tipo Samsung": "Samsung Galaxy Watch 3", "Tipo Xiaomi / Amazfit": "Xiaomi Mi Band 5",
            "Tipo Garmin": "Garmin Vivosmart 4", "Peor caso (Withings)": "Withings ScanWatch"}
params = {}
for name, src in SCEN_SRC.items():
    r = lit.loc[src]
    params[name] = dict(dT=r["sesgo TST (min)"] - base["sesgo TST (min)"],
                        dS=0.0 if np.isnan(r["sesgo eficiencia (pts)"]) else r["sesgo eficiencia (pts)"] - base["sesgo eficiencia (pts)"],
                        dW=0.0 if np.isnan(r["sesgo WASO (min)"]) else r["sesgo WASO (min)"] - base["sesgo WASO (min)"],
                        noise=float(np.sqrt(max(r["DE de las diferencias TST (min)"] ** 2 - sd_fit ** 2, 0))))
pd.DataFrame(params).T.round(1).rename(columns={"dT": "Δ tiempo dormido (min)", "dS": "Δ eficiencia (pts)", "dW": "Δ min. despierto", "noise": "ruido extra por noche (min)"})

,Δ tiempo dormido (min),Δ eficiencia (pts),Δ min. despierto,ruido extra por noche (min)
Tipo Samsung,0.8,0.0,0.0,0.0
Tipo Xiaomi / Amazfit,20.8,3.0,-16.8,48.6
Tipo Garmin,29.7,5.4,-23.7,29.6
Peor caso (Withings),31.2,7.4,-33.3,40.7


## 2. Tu punto 3: ¿se pueden aproximar los minutos sedentarios?

Health Connect **no** tiene minutos sedentarios, pero sí pasos, distancia, calorías y pulso. Se entrena un modelo **por día** que
predice los minutos sedentarios de Fitbit a partir de esas variables, siempre **sobre personas que el modelo no vio** (5 grupos
de personas). Cuatro conjuntos de entrada:

- S1: solo pasos · S2: pasos, distancia y calorías · S3: S2 más el pulso medio diario
- S4 (**hipotético**): S3 más minutos de actividad moderada y vigorosa; Health Connect podría entregarlos en versiones recientes
  de Android, pero no está confirmado y **no incluye minutos livianos**, que son los que más se relacionan con sedentarismo.

Luego se mide si agregar el sedentarismo aproximado al modelo recupera lo que se pierde al no tener el real.

In [4]:
SETS = {"S1: pasos": ["steps"],
        "S2: pasos + distancia + calorías": ["steps", "distance", "calories"],
        "S3: S2 + pulso medio diario": ["steps", "distance", "calories", "bpm"],
        "S4 (hipotético): S3 + min. moderados y vigorosos": ["steps", "distance", "calories", "bpm", "moderately_active_minutes", "very_active_minutes"]}
rows = []
for name, feats in SETS.items():
    d = daily.dropna(subset=feats + ["sedentary_minutes"])
    pred = pd.Series(np.nan, index=d.index)
    for tr, te in GroupKFold(5).split(d, groups=d[COL_ID]):
        m = RandomForestRegressor(150, max_depth=8, min_samples_leaf=20, random_state=SEED, n_jobs=-1).fit(d.iloc[tr][feats], d.iloc[tr]["sedentary_minutes"])
        pred.iloc[te] = m.predict(d.iloc[te][feats])
    daily.loc[pred.index, "sed_" + name[:2]] = pred
    rows.append({"conjunto": name, "días usados": len(d), "R² por día (personas no vistas)": round(r2_score(d.sedentary_minutes, pred), 2),
                 "Spearman por día": round(spearmanr(d.sedentary_minutes, pred).correlation, 2)})
pd.DataFrame(rows).set_index("conjunto")

,días usados,R² por día (personas no vistas),Spearman por día
conjunto,,,
S1: pasos,4777,0.20,0.44
S2: pasos + distancia + calorías,4777,0.18,0.43
S3: S2 + pulso medio diario,4727,0.29,0.53
S4 (hipotético): S3 + min. moderados y vigorosos,4727,0.33,0.56


In [5]:
# ¿el sedentarismo de Fitbit refleja el tiempo que se lleva puesto el reloj o el sueño, más que el movimiento?
d37 = daily[daily[COL_ID].isin(ids37)].copy()
d37["min_registrados"] = d37[["sedentary_minutes", "lightly_active_minutes", "moderately_active_minutes", "very_active_minutes"]].sum(axis=1, min_count=4)
pm = d37.groupby(COL_ID)[["sedentary_minutes", "min_registrados", "minutesAsleep", "steps"]].mean().reindex(ids37)
print("Correlación de Spearman entre personas (37) del promedio de minutos sedentarios con:")
for c, lab in [("min_registrados", "minutos totales registrados (tiempo con reloj puesto)"), ("minutesAsleep", "minutos dormidos"), ("steps", "pasos")]:
    print(f"  {lab:55s} {spearmanr(pm.sedentary_minutes, pm[c], nan_policy='omit').correlation:+.2f}")
print("Correlación del promedio de sedentarios con el índice de riesgo:", f"{spearmanr(pm.sedentary_minutes, y).correlation:+.2f}")
print("Correlación de los minutos totales registrados con el índice de riesgo:", f"{spearmanr(pm.min_registrados, y).correlation:+.2f}")


def _resid(a, b):
    Z = np.c_[np.ones(len(b)), b]
    return a - Z @ np.linalg.lstsq(Z, a, rcond=None)[0]


rk = pd.DataFrame({"sed": pm.sedentary_minutes, "reg": pm.min_registrados, "y": y}).rank()
print("Correlación parcial riesgo ~ sedentarios, quitando el efecto de los minutos registrados:",
      f"{np.corrcoef(_resid(rk.y.values, rk.reg.values), _resid(rk.sed.values, rk.reg.values))[0, 1]:+.2f}")
print("Correlación parcial riesgo ~ minutos registrados, quitando el efecto de los sedentarios:  ",
      f"{np.corrcoef(_resid(rk.y.values, rk.sed.values), _resid(rk.reg.values, rk.sed.values))[0, 1]:+.2f}")
print(f"Minutos registrados por día: promedio {d37.min_registrados.mean():.0f} de 1440 | días con menos de 1000: {(d37.min_registrados < 1000).mean():.0%}")

Correlación de Spearman entre personas (37) del promedio de minutos sedentarios con:
  minutos totales registrados (tiempo con reloj puesto)   +0.88
  minutos dormidos                                        -0.15
  pasos                                                   -0.10
Correlación del promedio de sedentarios con el índice de riesgo: +0.70
Correlación de los minutos totales registrados con el índice de riesgo: +0.68
Correlación parcial riesgo ~ sedentarios, quitando el efecto de los minutos registrados: +0.29
Correlación parcial riesgo ~ minutos registrados, quitando el efecto de los sedentarios:   +0.20
Minutos registrados por día: promedio 1178 de 1440 | días con menos de 1000: 31%


## 3. Modelo con variables que sí entrega cada reloj: ¿se recupera con el sedentarismo aproximado?
Mismo protocolo de siempre (5-Fold × 20 repeticiones, 37 personas, sin distorsión). El "sedentarismo real" es solo referencia:
no estará disponible fuera de Fitbit.

In [6]:
def feat_all(df, ids, with_sed=()):
    g = df[df[COL_ID].isin(ids)].groupby(COL_ID)
    cols = {}
    for v in ["minutesAsleep", "sleep_efficiency", "minutesAwake", "resting_hr", "steps", "sedentary_minutes", "spo2", "rmssd", "nremhr",
              "full_sleep_breathing_rate", "minutesToFallAsleep"] + list(with_sed):
        cols[f"{v}_mean"], cols[f"{v}_std"] = g[v].mean(), g[v].std()
    return pd.DataFrame(cols).reindex(ids)


SEDCOLS = ["sed_" + n[:2] for n in SETS]
X_clean = feat_all(daily, ids37, with_sed=SEDCOLS)
med = X_clean.median()
CORE = ["minutesAsleep_mean", "minutesAsleep_std", "sleep_efficiency_mean", "sleep_efficiency_std", "resting_hr_mean", "resting_hr_std", "steps_mean", "steps_std"]


def fit_rf(X_, y_):
    return RandomForestRegressor(300, max_depth=5, random_state=SEED, n_jobs=-1).fit(X_, y_)


def plain_cv(cols):
    X_ = X_clean[cols].fillna(med[cols]); out = []
    for r in range(20):
        p = np.zeros(len(X_))
        for tr, te in KFold(5, shuffle=True, random_state=SEED + r).split(X_):
            p[te] = fit_rf(X_.iloc[tr], y.iloc[tr]).predict(X_.iloc[te])
        out.append({"spearman": spearmanr(y, p).correlation, "roc": roc_auc_score(y_alto, p)})
    o = pd.DataFrame(out); return o.mean(), o.std(ddof=0)


variants = {"Núcleo común (sin sedentarios)": CORE,
            "+ sedentarios REALES (solo referencia)": CORE + ["sedentary_minutes_mean", "sedentary_minutes_std"]}
for n_ in SETS:
    variants[f"+ sedentarios aproximados {n_[:2]}"] = CORE + [f"sed_{n_[:2]}_mean", f"sed_{n_[:2]}_std"]
res2 = {k: plain_cv(v) for k, v in variants.items()}
tab2 = pd.DataFrame({k: {"Spearman": f"{m['spearman']:.3f} ± {s['spearman']:.3f}", "ROC-AUC (Alto)": f"{m['roc']:.3f} ± {s['roc']:.3f}"} for k, (m, s) in res2.items()}).T
tab2

,Spearman,ROC-AUC (Alto)
Núcleo común (sin sedentarios),0.395 ± 0.087,0.750 ± 0.048
+ sedentarios REALES (solo referencia),0.661 ± 0.051,0.792 ± 0.033
+ sedentarios aproximados S1,0.525 ± 0.051,0.793 ± 0.022
+ sedentarios aproximados S2,0.441 ± 0.074,0.764 ± 0.047
+ sedentarios aproximados S3,0.506 ± 0.053,0.757 ± 0.033
+ sedentarios aproximados S4,0.449 ± 0.070,0.762 ± 0.040


## 4. Datos sintéticos por marca: ¿cuánto se degrada el modelo generalizado?

Para cada escenario de marca se **distorsionan los datos diarios de las 37 personas**: al tiempo dormido se le suma el sesgo,
a la eficiencia y a los minutos despierto también, y opcionalmente se agrega ruido por noche. Después se recalculan las
variables por persona. El modelo **se entrena con los datos sin distorsión (Fitbit)** y se prueba con los distorsionados,
siempre en personas que no vio.

**Diseños de variables comparados**
- **Núcleo común:** tiempo dormido, eficiencia, pulso en reposo y pasos (media y variación de cada uno).
- **Sin eficiencia:** el núcleo sin la eficiencia (la más sensible a la definición de "despierto").
- **Robusto:** solo la **variación** del tiempo dormido (un sesgo constante no la cambia), más pulso y pasos.
- **Robusto + sedentarios aproximados:** el anterior más el mejor sedentarismo aproximado de la sección 2.
- **Completo (referencia):** las 11 variables de Fitbit.

Métricas: Spearman (¿sigue ordenando bien?), ROC-AUC del nivel Alto, **acierto del nivel exacto** (Leve / Moderado / Alto) y
**deriva** (cuánto se desplaza el índice medio, en puntos tipo PSQI). Lo que más importa es la deriva, porque un sesgo constante
no cambia el orden pero sí mueve a la gente de nivel.

In [7]:
best_sed = max([n_ for n_ in SETS if not n_.startswith("S4")], key=lambda n_: res2[f"+ sedentarios aproximados {n_[:2]}"][0]["spearman"])
b = "sed_" + best_sed[:2]
print("Sedentarismo aproximado elegido para el diseño robusto:", best_sed)
ALLV = ["minutesAsleep", "sleep_efficiency", "minutesAwake", "resting_hr", "steps", "sedentary_minutes", "spo2", "rmssd", "nremhr",
        "full_sleep_breathing_rate", "minutesToFallAsleep"]
VARIANTS = {
    "Completo (referencia)": [f"{v}_{s_}" for v in ALLV for s_ in ("mean", "std")],
    "Núcleo común": CORE,
    "Sin eficiencia": [c for c in CORE if not c.startswith("sleep_efficiency")],
    "Robusto": ["minutesAsleep_std", "resting_hr_mean", "resting_hr_std", "steps_mean", "steps_std"],
    "Robusto + sedentarios aprox.": ["minutesAsleep_std", "resting_hr_mean", "resting_hr_std", "steps_mean", "steps_std", f"{b}_mean", f"{b}_std"],
}


def distort(df, dT, dS, dW, noise, seed):
    rng = np.random.default_rng(seed); d = df.copy()
    n_ = len(d)
    d["minutesAsleep"] = (d["minutesAsleep"] + dT + rng.normal(0, noise, n_)).clip(lower=0)
    d["sleep_efficiency"] = (d["sleep_efficiency"] + dS).clip(upper=100)
    d["minutesAwake"] = (d["minutesAwake"] + dW).clip(lower=0)
    return d


SCEN = {"Sin distorsión (Fitbit)": None}
for name, p_ in params.items():
    SCEN[f"{name}: solo sesgo"] = dict(p_, noise=0.0)
    SCEN[f"{name}: sesgo + ruido"] = dict(p_)
X_scen = {}
for i, (s_, p_) in enumerate(SCEN.items()):
    X_scen[s_] = X_clean if p_ is None else feat_all(distort(daily, seed=SEED + i, **p_), ids37, with_sed=SEDCOLS)


def metrics(p):
    lv = pd.Series([risk_level(v) for v in p]); tv = pd.Series([risk_level(v) for v in y])
    return {"spearman": spearmanr(y, p).correlation, "roc": roc_auc_score(y_alto, p), "nivel_exacto": float((lv.values == tv.values).mean()),
            "deriva": float((np.mean(p) - y.mean()) * SCALE)}


R = {v: {s_: [] for s_ in SCEN} for v in VARIANTS}
for v, cols in VARIANTS.items():
    Xtr = X_clean[cols].fillna(med[cols]); Xte = {s_: X_scen[s_][cols].fillna(med[cols]) for s_ in SCEN}
    for r in range(20):
        ps = {s_: np.zeros(len(Xtr)) for s_ in SCEN}
        for tr, te in KFold(5, shuffle=True, random_state=SEED + r).split(Xtr):
            m = fit_rf(Xtr.iloc[tr], y.iloc[tr])
            for s_ in SCEN:
                ps[s_][te] = m.predict(Xte[s_].iloc[te])
        for s_ in SCEN:
            R[v][s_].append(metrics(ps[s_]))


def pivot(metric, fmt):
    return pd.DataFrame({v: {s_: fmt(np.mean([x[metric] for x in R[v][s_]])) for s_ in SCEN} for v in VARIANTS})


print("Spearman (más alto = mejor orden)"); display(pivot("spearman", lambda x: f"{x:.2f}"))
print("Acierto del nivel exacto Leve/Moderado/Alto (con Alto real = %d de 37)" % y_alto.sum()); display(pivot("nivel_exacto", lambda x: f"{x:.0%}"))
print("Deriva del índice medio, en puntos tipo PSQI (0 = sin desplazamiento)"); display(pivot("deriva", lambda x: f"{x:+.1f}"))

Sedentarismo aproximado elegido para el diseño robusto: S1: pasos


Spearman (más alto = mejor orden)


,Completo (referencia),Núcleo común,Sin eficiencia,Robusto,Robusto + sedentarios aprox.
Sin distorsión (Fitbit),0.70,0.40,0.46,0.41,0.53
Tipo Samsung: solo sesgo,0.70,0.40,0.46,0.41,0.53
Tipo Samsung: sesgo + ruido,0.70,0.40,0.46,0.41,0.53
Tipo Xiaomi / Amazfit: solo sesgo,0.70,0.43,0.47,0.41,0.53
Tipo Xiaomi / Amazfit: sesgo + ruido,0.70,0.44,0.52,0.44,0.52
Tipo Garmin: solo sesgo,0.69,0.36,0.46,0.41,0.53
Tipo Garmin: sesgo + ruido,0.69,0.41,0.50,0.44,0.53
Peor caso (Withings): solo sesgo,0.68,0.36,0.45,0.41,0.53
Peor caso (Withings): sesgo + ruido,0.69,0.41,0.54,0.48,0.53


Acierto del nivel exacto Leve/Moderado/Alto (con Alto real = 11 de 37)


,Completo (referencia),Núcleo común,Sin eficiencia,Robusto,Robusto + sedentarios aprox.
Sin distorsión (Fitbit),61%,58%,59%,59%,56%
Tipo Samsung: solo sesgo,61%,59%,60%,59%,56%
Tipo Samsung: sesgo + ruido,61%,59%,60%,59%,56%
Tipo Xiaomi / Amazfit: solo sesgo,59%,60%,59%,59%,56%
Tipo Xiaomi / Amazfit: sesgo + ruido,59%,58%,58%,58%,56%
Tipo Garmin: solo sesgo,59%,57%,60%,59%,56%
Tipo Garmin: sesgo + ruido,59%,58%,58%,59%,56%
Peor caso (Withings): solo sesgo,59%,56%,60%,59%,56%
Peor caso (Withings): sesgo + ruido,59%,58%,59%,59%,56%


Deriva del índice medio, en puntos tipo PSQI (0 = sin desplazamiento)


,Completo (referencia),Núcleo común,Sin eficiencia,Robusto,Robusto + sedentarios aprox.
Sin distorsión (Fitbit),-0.1,-0.3,-0.3,-0.3,-0.2
Tipo Samsung: solo sesgo,-0.1,-0.3,-0.3,-0.3,-0.2
Tipo Samsung: sesgo + ruido,-0.1,-0.3,-0.3,-0.3,-0.2
Tipo Xiaomi / Amazfit: solo sesgo,-0.1,-0.6,-0.5,-0.3,-0.2
Tipo Xiaomi / Amazfit: sesgo + ruido,-0.0,-0.0,+0.1,+0.3,-0.0
Tipo Garmin: solo sesgo,-0.2,-0.7,-0.6,-0.3,-0.2
Tipo Garmin: sesgo + ruido,-0.1,-0.5,-0.3,-0.1,-0.2
Peor caso (Withings): solo sesgo,-0.1,-0.7,-0.6,-0.3,-0.2
Peor caso (Withings): sesgo + ruido,-0.0,-0.2,-0.0,+0.2,-0.0


In [8]:
# ROC-AUC del nivel Alto: resumen por diseño (mínimo y media entre los 9 escenarios) frente al caso sin distorsión
rows = []
for v in VARIANTS:
    rocs = {s_: np.mean([x["roc"] for x in R[v][s_]]) for s_ in SCEN}
    nvl = {s_: np.mean([x["nivel_exacto"] for x in R[v][s_]]) for s_ in SCEN}
    drift = {s_: abs(np.mean([x["deriva"] for x in R[v][s_]])) for s_ in SCEN}
    ref = rocs["Sin distorsión (Fitbit)"]
    rows.append({"diseño": v, "ROC-AUC sin distorsión": round(ref, 2), "ROC-AUC peor escenario": round(min(rocs.values()), 2),
                 "nivel exacto sin distorsión": f"{nvl['Sin distorsión (Fitbit)']:.0%}", "nivel exacto peor escenario": f"{min(nvl.values()):.0%}",
                 "deriva absoluta máx. (pts PSQI)": round(max(drift.values()), 1)})
pd.DataFrame(rows).set_index("diseño")

,ROC-AUC sin distorsión,ROC-AUC peor escenario,nivel exacto sin distorsión,nivel exacto peor escenario,deriva absoluta máx. (pts PSQI)
diseño,,,,,
Completo (referencia),0.83,0.82,61%,59%,0.2
Núcleo común,0.75,0.75,58%,56%,0.7
Sin eficiencia,0.80,0.80,59%,58%,0.6
Robusto,0.76,0.76,59%,58%,0.3
Robusto + sedentarios aprox.,0.79,0.79,56%,56%,0.2


## Lectura de los resultados (exploratorio)

**1. Los sesgos conocidos de cada marca casi no degradan el modelo generalizado.** Aplicando a los datos de Fitbit los
sesgos de Xiaomi, Garmin y hasta el peor caso de la literatura (Withings), el orden que hace el modelo casi no cambia
(Spearman entre 0.36 y 0.44 contra 0.40 sin distorsión), el acierto del nivel exacto se mantiene entre 56 % y 61 %, y la deriva
del índice medio no pasa de 0.7 puntos tipo PSQI. **Cuidado con la razón:** no es que el modelo sea "robusto" en general, sino
que **casi no usa las variables de sueño** (su fuerza viene de actividad y pulso), y los sesgos de 20 a 30 minutos son pequeños
frente a la variación entre personas (unos 90 minutos). Con el PSQI real, donde el sueño sí pesará, este resultado puede cambiar.
No se puede dar por sentado.

**2. Lo que sí cuesta generalizar no es el sesgo de los relojes: es perder los minutos sedentarios.** El núcleo común sin ellos
queda en Spearman 0.40 (ROC-AUC 0.75); con los reales sube a 0.66 (0.79). Los diseños "sin eficiencia" y "robusto" (solo variación
del sueño, pulso y pasos) rinden parecido al núcleo común y **sufren igual o menos con los sesgos**, así que prescindir de
eficiencia y minutos despierto **no cuesta nada aquí** y protege de definiciones distintas entre marcas (recordar que Xiaomi
promedió 6.6 min despierto contra 50.6 en Fitbit).

**3. Aproximar los minutos sedentarios recupera algo, pero poco y sin consistencia.** Con pasos, distancia, calorías y pulso el
modelo por día explica solo de 18 % a 33 % de la variación (Spearman por día de 0.43 a 0.56). Al agregar el sedentarismo
aproximado al modelo, el Spearman pasa de 0.40 a entre 0.44 y 0.53, pero **el mejor conjunto es el más simple (solo pasos)** y
los más completos rinden menos: es señal de ruido, no de una mejora real. Con solo 37 personas no se puede decir que la
aproximación funcione. Y lo que sigue es más importante.

**4. Alerta: el "factor principal" del modelo está enredado con cuánto tiempo registró el reloj.** El promedio de minutos
sedentarios correlaciona **+0.88** con los minutos totales registrados (el tiempo que el reloj estuvo puesto), y el riesgo del
proxy correlaciona +0.68 con esos minutos registrados casi igual que con los sedentarios (+0.70). Al quitar el efecto de uno
sobre el otro, la relación cae a +0.29 (sedentarios) y +0.20 (minutos registrados). O sea, **buena parte de la señal más fuerte
del modelo no se puede separar de qué tan completos están los datos de cada persona**. Puede reflejar comportamiento real,
un artefacto del puntaje sustituto (que Fitbit calcula con la misma actividad y sueño), o ambos. Hasta separarlos, **no debería
presentarse "sedentarismo" como hallazgo ni como factor principal en el LLM**, y este es otro motivo para reemplazar el proxy
por el PSQI.

**Qué implica para el modelo generalizado**
- Un núcleo común sin eficiencia ni minutos despierto es una apuesta razonable y barata: no pierde y es menos sensible a
  definiciones distintas. Pero hay que **repetir esta prueba con el PSQI**, donde el sueño sí importará.
- No conviene apostar por aproximar sedentarios ni tratar el sedentarismo como el factor central hasta separarlo del tiempo de
  registro.
- **Nada aquí sustituye a datos reales** de los relojes (los que están llegando): la misma persona con dos relojes o, al menos,
  los mismos participantes durante varias noches.

**Límites:** los datos por marca son copias distorsionadas de datos reales de Fitbit; los sesgos vienen de estudios con 20 a 45
personas, con signos y convenciones que hubo que ajustar para Xiaomi; para Amazfit no hay validación y se supuso igual a Xiaomi;
para Samsung solo hay sesgo de tiempo dormido; no se modelaron errores de pasos ni de pulso (no hay fuente comparable); y todo
usa el puntaje sustituto con n = 37, cuya variación entre repeticiones (± 0.05 a 0.09) solo refleja cómo se reparten las
personas, no la incertidumbre de haber medido a 37.
